In [ ]:
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
from IPython.display import display
from src.research_config import ResearchConfig
from src.research_data import prepare_prices


# 01 Market Data
Download the S&P 500 constituents, adjusted stock prices, S&P 500 benchmark and 13-week Treasury yield in one module, then create the chronological 70/30 split. The constituent list is a current snapshot, so survivorship bias remains.


In [ ]:
cfg = ResearchConfig()
START_DATE = "2016-01-01"
END_DATE = "2026-01-01"
CONSTITUENTS_URL = "https://raw.githubusercontent.com/datasets/s-and-p-500-companies/master/data/constituents.csv"
display(pd.Series(cfg.to_dict(), name="Research settings"))


## Download market data
Stocks use adjusted closes. `^GSPC` supplies the market benchmark and session calendar. `^IRX` is used as the annual risk-free yield proxy after division by 100.


In [ ]:
constituents = pd.read_csv(CONSTITUENTS_URL)
tickers = sorted(constituents["Symbol"].str.replace(".", "-", regex=False).unique())

download = yf.download(
    tickers + ["^GSPC", "^IRX"],
    start=START_DATE,
    end=END_DATE,
    auto_adjust=False,
    progress=True,
    threads=False,
)
adjusted = download["Adj Close"].sort_index()
close = download["Close"].sort_index()

sessions = adjusted["^GSPC"].dropna().index
prices = adjusted.reindex(index=sessions, columns=tickers)
benchmark = close["^GSPC"].reindex(sessions)
risk_free_rates = close["^IRX"].dropna().sort_index() / 100

display(prices.head())
display(risk_free_rates.describe())


## Formation and out-of-sample split
Asset availability is assessed only in the first 70% of sessions. Formation gaps are forward-filled from prior observations; the test panel is left unchanged.


In [ ]:
train_prices, test_prices, availability = prepare_prices(prices, cfg)

display(pd.DataFrame({
    "observations": [len(train_prices), len(test_prices)],
    "start": [train_prices.index.min(), test_prices.index.min()],
    "end": [train_prices.index.max(), test_prices.index.max()],
}, index=["Formation", "Out of sample"]))
display(availability.head(10))


## Save data for the remaining modules


In [ ]:
constituents.to_csv("constituents.csv", index=False)
train_prices.to_parquet("train_prices.parquet")
test_prices.to_parquet("test_prices.parquet")
availability.to_parquet("availability.parquet")
benchmark.to_frame("benchmark").to_parquet("benchmark_prices.parquet")
risk_free_rates.to_frame("risk_free_rate").to_parquet("risk_free_rates.parquet")

(train_prices.iloc[:, :5] / train_prices.iloc[0, :5]).plot(
    figsize=(10, 4), title="Formation prices normalized to one"
)
plt.show()
risk_free_rates.plot(figsize=(10, 3), title="13-week Treasury annual yield proxy")
plt.show()
